## Part 1: Load and align financial data

The goal of this notebook will be to load financial data from FRED (Federal Reserve Bank of St. Louis) and Yahoo Finance which will be used to train models in an attempt to predict future market health. The data will also be pre-processed, meaning that data will be aligned, sampling frequencies matched and features adjusted to better match real-world conditions. The pre-processing steps are outlined below.

In [38]:
import numpy as np
import pandas as pd
import datetime

from macro_etf.data_loader import load_yfinance_data, load_fred_data
from macro_etf.common import PROCESSED_DATA_DIR

### 1.1: FRED data

Data is loaded using the fredapi Python package, which requires an API and downloads data from [fred.stlouisfed.org](https://fred.stlouisfed.org/docs/api/api_key.html). These data correspond to broad market health indicators which I will use together with data retrieved from Yahoo Finance (see 1.2 below) to measure overall market health. A description for each feature pulled from FRED can be found in [DATA.md](../DATA.md).

Let's start by loading the data and taking a look at it in it's raw state.

In [39]:
df_fred = load_fred_data()
df_fred.describe()

,cpi,unemployment,yield_spread,fed_funds_rate,credit_spread,financial_stress,industrial_production,retail_sales,real_gdp,consumer_sentiment,credit_card_delinquency,personal_savings_rate,housing_starts,building_permits,m2,initial_claims,job_openings
count,341.000000,341.000000,7142.000000,342.000000,7135.000000,1490.000000,342.000000,342.000000,113.000000,341.000000,113.000000,341.000000,342.000000,342.000000,341.000000,1.490000e+03,306.000000
mean,230.075663,5.530205,0.988853,2.257690,2.429271,0.029175,96.626800,379275.154971,17968.199628,83.100293,3.515929,5.672727,1331.903509,1389.219298,11422.640469,3.627094e+05,5498.856209
std,45.098990,1.893549,0.957252,2.111228,0.727992,1.063412,5.251040,122684.760405,3080.990971,15.096111,1.217496,3.042433,413.940239,424.785993,5862.349475,3.315037e+05,2275.757305
min,162.000000,3.400000,-1.080000,0.050000,1.360000,-1.128300,84.321800,204207.000000,12703.742000,44.800000,1.530000,1.400000,478.000000,513.000000,4056.200000,1.870000e+05,2232.000000
25%,192.400000,4.200000,0.190000,0.160000,1.865000,-0.598250,92.341650,289787.000000,15844.727000,71.700000,2.490000,4.300000,1073.250000,1100.000000,6438.900000,2.450000e+05,3769.500000
50%,228.524000,4.900000,0.850000,1.695000,2.290000,-0.205050,98.693900,348000.000000,17367.010000,84.900000,3.220000,5.400000,1362.000000,1446.500000,9853.900000,3.160000e+05,4734.500000
75%,255.213000,6.100000,1.820000,4.330000,2.840000,0.313875,100.955075,437901.500000,20304.874000,95.000000,4.570000,6.200000,1621.500000,1670.500000,14594.700000,3.870000e+05,7003.500000
max,333.979000,14.800000,2.910000,6.540000,6.160000,9.675300,104.100400,666056.000000,24180.419000,112.000000,6.770000,31.800000,2273.000000,2263.000000,23052.300000,6.137000e+06,12301.000000


In [40]:
df_fred.head()

,cpi,unemployment,yield_spread,fed_funds_rate,credit_spread,financial_stress,industrial_production,retail_sales,real_gdp,consumer_sentiment,credit_card_delinquency,personal_savings_rate,housing_starts,building_permits,m2,initial_claims,job_openings
1998-01-01,162.0,4.6,NaN,5.56,NaN,NaN,84.3218,204505.0,12703.742,106.6,4.76,7.0,1525.0,1555.0,4056.2,NaN,NaN
1998-01-02,NaN,NaN,0.08,NaN,1.54,-0.4704,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1998-01-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,312000.0,NaN
1998-01-05,NaN,NaN,0.05,NaN,1.60,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1998-01-06,NaN,NaN,0.10,NaN,1.63,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


We can see that there are many NaN values present in this dataset due to the different sampling rates of each variable. For example, yield_spread is sampled every weekday while real_gdp is sampled every quarter (3 months). We will have to aligned the variables by down sampling / up sampling variables such that there is a non-NAN value present in each field.

#### 1.1.1: Sync feature sample rates

First, let's determine how often each is sampled:

In [41]:
# Let's calculate the difference (delta) in sampling dates to determine how often each is sampled.
# Specifically, let's note the min, max and median of the deltas as that can tell us something 
# about the delta distribution. If min max and median are all the same, then the index is measured
# consistently. Otherwise it may be affected by holidays and such.
sample_data_delta_df = pd.DataFrame(index = df_fred.columns, columns = ['min', 'max', 'median', 'sample_rate'] )
for col in df_fred.columns:
    sample_dates = df_fred[col].dropna().index
    sample_date_deltas = [(sample_dates[i] - sample_dates[i-1]).days for i in range(1,len(sample_dates))]
    sample_data_delta_df.loc[col, 'min'] = np.min(sample_date_deltas)
    sample_data_delta_df.loc[col, 'max'] = np.max(sample_date_deltas)
    sample_data_delta_df.loc[col, 'median'] = np.median(sample_date_deltas)
    
    #If sample date delta median is between 28 and 31, then sampling rate is monthly.
    if sample_data_delta_df.loc[col, 'median'] >= 28 and sample_data_delta_df.loc[col, 'median'] <= 31:
        sample_data_delta_df.loc[col, 'sample_rate'] = 'monthly'
    #If sample date delta min and max are 1, then sampling rate is daily with no exceptions.
    elif sample_data_delta_df.loc[col, 'min'] == 1 and sample_data_delta_df.loc[col, 'max']==1:
        sample_data_delta_df.loc[col, 'sample_rate'] = 'daily'
    #If sample date delta median is 1 and max > 1, then I will say that sampling is done every weekday. 
    #This may not be strictly true but okay for this project.
    elif sample_data_delta_df.loc[col, 'median'] == 1 and sample_data_delta_df.loc[col, 'max']>1:
        sample_data_delta_df.loc[col, 'sample_rate'] = 'weekdaily'
    #If sample date delta median is 7, then sampling rate is weekly.
    elif sample_data_delta_df.loc[col, 'median'] == 7:
        sample_data_delta_df.loc[col, 'sample_rate'] = 'weekly'
    #If sample date delta median is over 90, then sampling rate is quarterly.
    elif sample_data_delta_df.loc[col, 'median'] >90:
        sample_data_delta_df.loc[col, 'sample_rate'] = 'quarterly'
    

sample_data_delta_df


,min,max,median,sample_rate
cpi,28,61,31.0,monthly
unemployment,28,61,31.0,monthly
yield_spread,1,4,1.0,weekdaily
fed_funds_rate,28,31,31.0,monthly
credit_spread,1,5,1.0,weekdaily
financial_stress,7,7,7.0,weekly
industrial_production,28,31,31.0,monthly
retail_sales,28,31,31.0,monthly
real_gdp,90,92,91.5,quarterly
consumer_sentiment,28,31,31.0,monthly


For CPI and unemployement, why is the maximum delta two months? That implies a gap in the data which I will need to address.

In [42]:
has_month_skip = ['cpi', 'unemployment']
for col in has_month_skip:
    sample_dates = df_fred[col].dropna().index
    sample_date_deltas = [(sample_dates[i] - sample_dates[i-1]).days for i in range(1,len(sample_dates))]
    where_2month_skip = np.arange(len(sample_date_deltas))[np.isclose(sample_date_deltas,61)][0]
    print(sample_dates[where_2month_skip])

2025-09-01 00:00:00
2025-09-01 00:00:00


From this we can see that the gap in monthly cpi and unemployment records occurred in Oct 2025 (Sept 2025 is the date before the 2 month gap).
A quick google search shows that this occurred because of a prolonged government shutdown, resulting in the first ever gap in these records. I will correct for this by filling in a value for the Oct 2025 date using linear inference (i.e. take the mean of the values before and after Oct 2025).

In [43]:
previous_month = datetime.datetime.fromisoformat("2025-09-01")
skipped_month = datetime.datetime.fromisoformat("2025-10-01")
next_month = datetime.datetime.fromisoformat("2025-11-01")
df_fred.loc[skipped_month, 'cpi'] = np.mean([df_fred.loc[[previous_month, next_month], 'cpi']]) 
df_fred.loc[skipped_month, 'unemployment'] = np.mean([df_fred.loc[[previous_month, next_month], 'unemployment']]) 

print(df_fred.loc[[previous_month, skipped_month, next_month], 'unemployment'])
print(df_fred.loc[[previous_month, skipped_month, next_month], 'cpi'])

2025-09-01    4.40
2025-10-01    4.45
2025-11-01    4.50
Name: unemployment, dtype: float64
2025-09-01    324.245
2025-10-01    324.654
2025-11-01    325.063
Name: cpi, dtype: float64


Okay, some are sampled daily, some weekly, some monthly and two quarterly. Let's align them.

First, let's decide on a rate to align to. I will choose monthly here, as that provides enough time for these measures to meaningfully move between sample dates, while not being so long as to make future predictions meaningless.

In [44]:
# Let's do the sampling using Pandas.Series.resample() method, which makes all of this nice and easy.

# Resample to month end
sync_to = 'ME' 

resampled_features = {}
# Downsampling from daily to monthly
    # These spreads are generally less spiked and so it should be okay to take the 
    # last value of the month, to capture current conditions at that date.
resampled_features['yield_spread'] =            df_fred['yield_spread'].resample(sync_to).last()
resampled_features['credit_spread'] =           df_fred['credit_spread'].resample(sync_to).last()
# resampled_features['ted_spread'] =              df_fred['ted_spread'].resample(sync_to).last()

# Downsampling from weekly to monthly
    # Take the mean here, as these are more variable and taking the mean allows us 
    # to capture any peaks occurring in the middle of the month.
resampled_features['financial_stress'] =        df_fred['financial_stress'].resample(sync_to).mean()
resampled_features['initial_claims'] =          df_fred['initial_claims'].resample(sync_to).mean()

# Correcting indices for features already sampled monthly.
resampled_features['cpi'] =                     df_fred['cpi'].resample(sync_to).last()
resampled_features['unemployment'] =            df_fred['unemployment'].resample(sync_to).last()
resampled_features['fed_funds_rate'] =          df_fred['fed_funds_rate'].resample(sync_to).last()
resampled_features['industrial_production'] =   df_fred['industrial_production'].resample(sync_to).last()
resampled_features['retail_sales'] =            df_fred['retail_sales'].resample(sync_to).last()
# resampled_features['leading_indicators'] =      df_fred['leading_indicators'].resample(sync_to).last()
resampled_features['consumer_sentiment'] =      df_fred['consumer_sentiment'].resample(sync_to).last()
resampled_features['personal_savings_rate'] =   df_fred['personal_savings_rate'].resample(sync_to).last()
resampled_features['housing_starts'] =          df_fred['housing_starts'].resample(sync_to).last()
resampled_features['building_permits'] =        df_fred['building_permits'].resample(sync_to).last()
resampled_features['m2'] =                      df_fred['m2'].resample(sync_to).last()
resampled_features['job_openings'] =            df_fred['job_openings'].resample(sync_to).last()

# Upsampling from quarterly to monthly.
    #ffill() copies the last valid entry into rows with missing values.
resampled_features['real_gdp'] =                df_fred['real_gdp'].resample(sync_to).first().ffill()
resampled_features['credit_card_delinquency'] = df_fred['credit_card_delinquency'].resample(sync_to).first().ffill()


View the resampled data in its current state

In [45]:
display(pd.DataFrame(resampled_features).head())
display(pd.DataFrame(resampled_features).tail())

,yield_spread,credit_spread,financial_stress,initial_claims,cpi,unemployment,fed_funds_rate,industrial_production,retail_sales,consumer_sentiment,personal_savings_rate,housing_starts,building_permits,m2,job_openings,real_gdp,credit_card_delinquency
1998-01-31,0.21,1.68,-0.144500,321200.0,162.0,4.6,5.56,84.3218,204505.0,106.6,7.0,1525.0,1555.0,4056.2,NaN,12703.742,4.76
1998-02-28,0.07,1.67,-0.353650,319000.0,162.0,4.6,5.51,84.4583,204207.0,110.4,7.0,1584.0,1647.0,4088.9,NaN,12703.742,4.76
1998-03-31,0.07,1.67,-0.325075,315000.0,162.0,4.7,5.49,84.4880,205311.0,106.5,7.1,1567.0,1605.0,4114.3,NaN,12703.742,4.76
1998-04-30,0.09,1.66,-0.222850,311000.0,162.2,4.3,5.45,84.7779,208683.0,108.7,6.9,1540.0,1547.0,4140.2,NaN,12821.339,4.76
1998-05-31,0.03,1.63,-0.197180,311200.0,162.6,4.4,5.49,85.3367,209165.0,106.5,6.6,1536.0,1554.0,4164.4,NaN,12821.339,4.76


,yield_spread,credit_spread,financial_stress,initial_claims,cpi,unemployment,fed_funds_rate,industrial_production,retail_sales,consumer_sentiment,personal_savings_rate,housing_starts,building_permits,m2,job_openings,real_gdp,credit_card_delinquency
2026-03-31,0.51,1.79,-0.315625,208000.000000,330.293,4.3,3.64,101.6172,653772.0,53.3,3.5,1522.0,1363.0,22686.2,6887.0,24180.419,2.92
2026-04-30,0.52,1.70,-0.575300,207750.000000,332.407,4.3,3.64,102.4196,657830.0,49.8,3.0,1414.0,1423.0,22804.5,7585.0,24180.419,2.92
2026-05-31,0.47,1.57,-0.720980,211600.000000,333.979,4.3,3.63,102.5606,664439.0,44.8,3.0,1199.0,1410.0,23052.3,7594.0,24180.419,2.92
2026-06-30,0.30,1.53,-0.808475,222500.000000,332.568,4.2,3.63,102.6395,666056.0,NaN,NaN,1427.0,1367.0,NaN,NaN,24180.419,2.92
2026-07-31,0.36,1.60,-0.769133,204333.333333,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24180.419,2.92


Notice that there are still some NaN values. Specifically, at the beginning of the dataframe there are no job_openings data and the last two rows contain several NaNs. 

The lack of data for job_openings is because that information wasn't tracked at that time, and the earliest data for it starts in Dec 2000. Due to this, in order to keep job openings as a part of our dataset, all rows corresponding to earlier dates will be removed below when I call df.drop_na().

The NaNs at the end of the DataFrame are caused by those values simply not being recorded and made public yet. Even for those that are reported monthly, it can take several months after the reported date to actually verify the numbers, causing delay. This is important to keep in mind, because that means that when defining our features, we will have to use values corresponding to what is known at the time. Several of these variables should thus be *shifted* relative to the dates so that the values represent what is known at that date.

Most shifts that need to happen only require a one month shift, as the processing delay is a few weeks to a month. job_openings has about a 5-week delay so that a 2 month shift is the most accurate. Finally, real_gdp is reported in phases. The advance estimate comes out about 4 weeks after quarters end which would translate to a 1 month lag. However, the refined version comes out about 2 months after quarters end meaning that a 2-3 month gap is more appropriate. I will go with a 2 month gap here.

In [46]:
# Shift some values to account for report delays. This can be done using pandas .shift()
# function, with the 'periods' argument set to how long of a lag there is between
# measurement date and measurement release.

release_lag_months = {
    'unemployment': 1,
    'industrial_production': 1,
    'retail_sales': 1,
    'consumer_sentiment': 1,
    'personal_savings_rate': 1,
    'housing_starts': 1,
    'building_permits': 1,
    'cpi': 1,
    'm2': 1,
    'credit_card_delinquency': 1, 
    'job_openings': 2,
    'real_gdp': 2
}

for feat, lag in release_lag_months.items():
    resampled_features[feat] = resampled_features[feat].shift(periods=lag)

In [47]:
new_fred_df = pd.DataFrame(resampled_features).dropna()
display(new_fred_df.head(5))
display(new_fred_df.tail(5))

,yield_spread,credit_spread,financial_stress,initial_claims,cpi,unemployment,fed_funds_rate,industrial_production,retail_sales,consumer_sentiment,personal_savings_rate,housing_starts,building_permits,m2,job_openings,real_gdp,credit_card_delinquency
2001-02-28,0.51,2.88,0.437625,371250.0,175.6,4.2,5.49,91.9020,247339.0,94.7,4.5,1600.0,1699.0,4978.3,5088.0,14229.765,4.81
2001-03-31,0.75,3.04,0.675540,387200.0,176.0,4.2,5.31,91.3034,247289.0,90.6,4.6,1625.0,1656.0,5017.0,5234.0,14183.120,4.81
2001-04-30,1.05,2.73,0.800700,396750.0,176.1,4.3,4.80,91.1162,244514.0,91.5,4.9,1590.0,1659.0,5074.8,5097.0,14183.120,4.81
2001-05-31,1.21,2.65,0.387725,394500.0,176.4,4.4,4.21,90.7891,249113.0,88.4,4.8,1649.0,1666.0,5139.1,4762.0,14183.120,4.94
2001-06-30,1.17,2.65,0.420820,397200.0,177.3,4.3,3.97,90.3555,250250.0,92.0,4.3,1605.0,1665.0,5137.2,4615.0,14271.694,4.94


,yield_spread,credit_spread,financial_stress,initial_claims,cpi,unemployment,fed_funds_rate,industrial_production,retail_sales,consumer_sentiment,personal_savings_rate,housing_starts,building_permits,m2,job_openings,real_gdp,credit_card_delinquency
2026-02-28,0.59,1.80,-0.545475,215750.0,326.588,4.3,3.64,101.0388,634949.0,56.4,4.4,1385.0,1393.0,22429.3,6550.0,24055.749,2.92
2026-03-31,0.51,1.79,-0.315625,208000.0,327.460,4.4,3.64,101.9263,641038.0,56.6,3.8,1346.0,1540.0,22626.9,7240.0,24180.419,2.92
2026-04-30,0.52,1.70,-0.575300,207750.0,330.293,4.3,3.64,101.6172,653772.0,53.3,3.5,1522.0,1363.0,22686.2,6922.0,24180.419,2.92
2026-05-31,0.47,1.57,-0.720980,211600.0,332.407,4.3,3.63,102.4196,657830.0,49.8,3.0,1414.0,1423.0,22804.5,6887.0,24180.419,2.92
2026-06-30,0.30,1.53,-0.808475,222500.0,333.979,4.3,3.63,102.5606,664439.0,44.8,3.0,1199.0,1410.0,23052.3,7585.0,24180.419,2.92


### 1.2: Load Yahoo Finance data

In [48]:
ticker_dfs = load_yfinance_data()

Let's merge the ticker data together and take a look at what information is available

In [49]:
ticker_df = pd.concat(ticker_dfs).unstack(level=0)
#Fix column names
ticker_df.columns = ticker_df.columns.to_series().map('_'.join)
ticker_df.head()

,Open_spy,Open_qqq,Open_vix,Open_oil,High_spy,High_qqq,High_vix,High_oil,Low_spy,Low_qqq,...,Dividends_vix,Dividends_oil,Stock Splits_spy,Stock Splits_qqq,Stock Splits_vix,Stock Splits_oil,Capital Gains_spy,Capital Gains_qqq,Capital Gains_vix,Capital Gains_oil
Date,,,,,,,,,,,,,,,,,,,,,
1998-01-02,59.555514,NaN,24.340000,NaN,59.765890,NaN,24.930000,NaN,59.077387,NaN,...,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,NaN,NaN
1998-01-05,59.880627,NaN,24.110001,NaN,60.244003,NaN,25.020000,NaN,59.230374,NaN,...,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,NaN,NaN
1998-01-06,59.517274,NaN,25.200001,NaN,59.536400,NaN,25.969999,NaN,58.867021,NaN,...,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,NaN,NaN
1998-01-07,58.809615,NaN,26.150000,NaN,59.192117,NaN,27.430000,NaN,58.274113,NaN,...,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,NaN,NaN
1998-01-08,58.943518,NaN,26.240000,NaN,58.943518,NaN,26.700001,NaN,58.369765,NaN,...,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,NaN,NaN


We can see that each ticker has data for:
- Open
- High
- Low
- Close
- Volume
- Dividends
- Stock Splits
- Capital gains

The first four of these are useful by themselves, though I will likely only make use of Close to track the value of each ticker at the end of the month.

Volume may be useful for tracking overall movement in the market, except for vix which as a measure is not traded and so has no volume.

Dividends I will ignore. These are the dividends that would be paid to you if you held the corresponding weighted mix of company stocks tracked by the index. By paying these dividends out instead of adding the value to the ETFs, the SPY and QQQ follow their respective indices more faithfully.

Stock splits may affect SPY and QQQ by causing their prices to jump up or down depending on the split. I will quickly check to see if there are any and then adjust the prices if necessary.

Capital gains, I will check to see if any are reported, though I suspect not as tickers are not for individual companies.

In [50]:
# Check if there are any dividends, stock splits or capital gains reported

for info_type in ['Stock Splits', 'Dividends', 'Capital Gains']:
    # print(info_type)
    for name, df in ticker_dfs.items():
        if info_type not in df.columns:
            continue
        if np.any(df[info_type]!=0):
            print(f'Ticker {name} contains non-zero entries in column {info_type}')

Ticker qqq contains non-zero entries in column Stock Splits
Ticker spy contains non-zero entries in column Dividends
Ticker qqq contains non-zero entries in column Dividends


Okay, so only SPY and QQQ contain dividends, which I am going to ignore, and so I will drop all three of these from the dataframe.

I am also going to drop open, high and low from the dataset and just use close to track each ticker.

In [51]:
columms_to_drop = [col for col in ticker_df.columns if 'Dividends' in col or
                                                        'Stock Splits' in col or 
                                                        'Capital Gains' in col or
                                                        'Open' in col or
                                                        'High' in col or
                                                        'Low' in col]
columms_to_drop.append("Volume_vix")

ticker_df = ticker_df.drop(columms_to_drop, axis='columns')
ticker_df.tail()


,Close_spy,Close_qqq,Close_vix,Close_oil,Volume_spy,Volume_qqq,Volume_oil
Date,,,,,,,
2026-07-17,743.289978,695.330017,18.770000,82.489998,62651000.0,54109300.0,96003.0
2026-07-20,742.090027,696.059998,18.650000,83.230003,50422300.0,29687600.0,99783.0
2026-07-21,748.280029,708.969971,17.049999,84.910004,34324300.0,33728300.0,298109.0
2026-07-22,747.409973,705.349976,16.639999,86.830002,32696300.0,23437600.0,298109.0
2026-07-23,737.039978,690.049988,19.400000,91.639999,36985575.0,32048188.0,361049.0


I will now need to downsample these as well, so that they can be merged with the FRED data, which is currently sampled monthly.

In [52]:
ticker_df = ticker_df.resample("ME").last()

### 1:3 Final touch up and then merge the datasets

With both the FRED data and yfinance data loaded and downsampled, I can merge them to create our final dataset for all future analysis.

Let's also move GDP to be the last column as I will use that as a measure of economic health and the last column is easily accessed / found

In [53]:
final_df = pd.concat([new_fred_df, ticker_df], axis=1, sort=True).dropna()

#Remove any capital letters if I missed any
final_df.columns = final_df.columns.to_series().str.lower().to_list()

#Move real_gdp to the last column as I will use that as an overall measure of economic health
# There is some controversy in this choice, but I will stick with it for now.
cols = final_df.columns.drop("real_gdp").insert(len(final_df.columns)-1,"real_gdp")
final_df = final_df.loc[:,cols]

final_df.to_csv(PROCESSED_DATA_DIR / 'processed_market_data.csv')